# LUMA - Lokalny urok mistrzostwa artystycznego 
LUMA to lokalny jarmark, który rośnie z tygodnia na tydzień, coraz więcej wystawców, więcej gości (miejscowych i turystó), okazjonalne promocje i sezonowość.

# Import bibliotek 

In [ ]:
import numpy as np
import pandas as pd
from itertools import chain
import matplotlib.pyplot as plt


# Wygenerowanie syntetycznych danych

## Ustawienie ziarna RNG


In [ ]:
RNG_SEED = 20261020
rng = np.random.default_rng(RNG_SEED)
print("Ustawiono ziarno: ", RNG_SEED)

## Zdefiniowanie "stałych'

In [ ]:
LUMA_FULL = "Lokalny Urok Mistrzostwa Artystycznego"
CATEGORIES = ["Ceramika", "Tekstylia", "Biżuteria", "Rękodzieło", "Przetwory", "Dekoracje", "Stolarka", "Masarstwo", "Alkohole"]

# Przygotowanie Dantch Syntetycznych

"Katalog" kategorii i ich bazowe parametry generatywne (Syntetyczne)

In [ ]:
catalog = pd.DataFrame({
    "category": CATEGORIES,
    "demand_loc_base": [120, 180, 130, 80, 160, 150, 70, 90, 110], # średni popyt lokalny (szt./tydz.)
    "demand_tur_base": [60, 40, 80, 30, 90, 30, 50, 40, 80], # średni popyt turystyczny (szt./tydz.)
    "price_pln_base": [8, 120, 150, 230, 40, 35, 250, 215, 135], # średnia cena jednostkowa (PLN/szt.)
})

Wyliczony "typowy" przychód tygodniowy: (lokalni + turyści) * cena

In [ ]:
catalog["revenue_avg_pln"] =(catalog["demand_loc_base"] + catalog["demand_tur_base"]) * catalog["price_pln_base"]

In [ ]:
catalog

### Przygotowanie funkcji pomocnicznych do generowania danych tygodniowych

Generowanie popytu lokalnego i turystycznego na podstawie danych bazowych, z losowym szumem (multiplikatywnym) i opcjonalnymi boostami (%)

In [ ]:
def determine_demand(catalog, rng, variance=0.15, boost_loc=0.0, boost_tur=0.0):
    df = catalog[["category", "demand_loc_base", "demand_tur_base"]].copy()
    eps_loc = np.clip(rng.normal(loc=1.0, scale=variance, size=len(df)), 0.5, 1.8)
    eps_tur = np.clip(rng.normal(loc=1.0, scale=variance, size=len(df)), 0.5, 1.8)
    df["demand_loc"] = (df["demand_loc_base"] * (1 + boost_loc)) * eps_loc
    df["demand_tur"] = (df["demand_tur_base"] * (1 + boost_tur)) * eps_tur 
    df["demand_loc"] = np.maximum(df["demand_loc"].round().astype(int), 0)
    df["demand_tur"] = np.maximum(df["demand_tur"].round().astype(int), 0)
    return df[["category", "demand_loc", "demand_tur"]]

Wyliczanie przychodu lokalnego, turystycznego i łacznego

In [ ]:
def count_revenue(catalog, demand_df):
    df = demand_df.merge(catalog[["category", "price_pln_base"]], on="category", how="left")
    df["revenue_loc"] = df["demand_loc"] * df["price_pln_base"]
    df["revenue_tur"] = df["demand_tur"] * df["price_pln_base"]
    df["revenue_pln"] = df["revenue_loc"] + df["revenue_tur"]
    return df[["category", "revenue_loc", "revenue_tur", "revenue_pln"]]

Generowanie danych tygodniowych

In [ ]:
def generate_weekly_data(catalog, rng, variance=0.15, boost_loc=0.0, boost_tur=0.0):
    demand_df = determine_demand(catalog, rng, variance=variance, boost_loc=boost_loc, boost_tur=boost_tur)
    revenue_df = count_revenue(catalog, demand_df)
    return revenue_df

### Wygenerowanie danych tygodniowych

In [ ]:
t1 = generate_weekly_data(catalog, rng)
t1 = t1.merge(catalog[["category", "price_pln_base"]], on="category", how="left")
print("Tydzień 1 - przychody per kategoria (PLN)")
display(t1)
print("Łączny przychód w tygodniu 1 (PLN)", t1["revenue_pln"].sum())

## Analiza i wizualizacja danych

### Podstawowe wykresy

In [ ]:
t1.plot.bar()

In [ ]:
t1[["revenue_loc", "revenue_tur"]].plot.bar()

In [ ]:
t1[["revenue_loc", "revenue_tur"]].plot.barh(stacked=True)

Minimalny wykres

In [ ]:
heights = t1["revenue_pln"]
plt.bar(range(len(heights)), heights)

Jawne tworzenie i wyświetlanie wykresu

In [ ]:
heights = t1["revenue_pln"]
categories = t1["category"]

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(categories, heights)
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(categories, heights)
plt.title("LUMA - Przychody per kategoria w tygodniu 1")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.show()

Etykiety wartości i formatowanie liczb

In [ ]:
def format_pln(x: float) -> str:
    return f"{int(round(x)):,} zł".replace(",", "\u202f")

In [ ]:
plt.figure(figsize=(12, 4))
bars = plt.bar(categories, heights)
ymin, ymax = plt.ylim() # pobranie wartości min i max dla y
pad = (ymax - ymin) * 0.07
plt.ylim(ymin, ymax + pad) # ustawienie nowej wartości y max
for bar in bars:
    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()
    plt.text(x, y, format_pln(y), ha="center", va="bottom")
plt.title("LUMA - Przychody per kategoria w tygodniu 1")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.tight_layout()
plt.show()

In [ ]:
def pad_axis(factor=0.07):
    ymin, ymax = plt.ylim() # pobranie wartości min i max dla y
    pad = (ymax - ymin) * factor
    plt.ylim(ymin, ymax + pad) # ustawienie nowej wartości y max

In [ ]:
def label_bars_pln(bars):
    for bar in bars:
        x = bar.get_x() + bar.get_width() / 2
        y = bar.get_height()
        plt.text(x, y, format_pln(y), ha="center", va="bottom")

### Formatowanie etykiet podziałki osi

In [ ]:
def format_tys(x: float, decimals: int = 0) -> str:
    v = x / 1000.0
    if decimals == 0:
        s = f"{int(round(v)):,}".replace(",", "\u202f")
    else:
        s = f"{v:,.{decimals}f}".replace(",", "\u202f").replace(".", ",")
    return s + " tys."

In [ ]:
def format_mln(x: float, decimals: int = 0) -> str:
    v = x / 1000000.0
    if decimals == 0:
        s = f"{int(round(v)):,}".replace(",", "\u202f")
    else:
        s = f"{v:,.{decimals}f}".replace(",", "\u202f").replace(".", ",")
    return s + " mln."

In [ ]:
plt.figure(figsize=(12, 4))
bars = plt.bar(categories, heights)
pad_axis()
label_bars_pln(bars)
ypos, ylab_old = plt.yticks()
ylab_new = [format_tys(y) for y in ypos]
plt.yticks(ypos, ylab_new)
plt.title("LUMA - Przychody per kategoria w tygodniu 1")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.tight_layout()
plt.show()

In [ ]:
def format_ytics_tys():
    ypos, _ = plt.yticks()
    ylab_new = [format_tys(y) for y in ypos]
    plt.yticks(ypos, ylab_new)

### Sortowanie kategorii dla łatwiejszego porównania

In [ ]:
t1_sorted = t1.sort_values("revenue_pln", ascending=False).reset_index()
categories = t1_sorted["category"]
heights = t1_sorted["revenue_pln"]

In [ ]:
plt.figure(figsize=(12, 4))
bars = plt.bar(categories, heights)
pad_axis()
label_bars_pln(bars)
format_ytics_tys()
plt.title("LUMA - Przychody per kategoria w tygodniu 1")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.tight_layout()
plt.show()

Siatka w tle wykresu

In [ ]:
plt.figure(figsize=(12, 4))
bars = plt.bar(categories, heights, color="powderblue")
pad_axis()
label_bars_pln(bars)
format_ytics_tys()
plt.grid(axis="y", linestyle=":", alpha=0.6)
plt.title("LUMA - Przychody per kategoria w tygodniu 1")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.tight_layout()
plt.show()

# Analiza i wizualizacja danych - 2 serie danych



### Wygenerowanie danych tygodniowych dla Tygodnia 2

Mini-festiwal przyciąga klientów: lokalni +13%, turyści +20%

In [ ]:
boost_loc = 0.13
boost_tur = 0.20

t2 = generate_weekly_data(catalog, rng, boost_loc=boost_loc, boost_tur=boost_tur)
t2 = t2.merge(catalog[["category", "price_pln_base"]], on="category", how="left")
print("Tydzień 2 - przychody per kategoria (PLN)")
display(t2)
print("Łączny przychód w tygodniu 1: ", format_pln(t1["revenue_pln"].sum()))
print("Łączny przychód w tygodniu 2: ", format_pln(t2["revenue_pln"].sum()))


Przygotowanie danych do wykresów

In [ ]:
categories = t1_sorted["category"]
t1_revenue_total = t1_sorted["revenue_pln"]
t2_revenue_total = t2.set_index("category").loc[categories]["revenue_pln"]
x_locs = np.arange(len(categories))
bar_width = 0.4
bar_half = bar_width / 2

In [ ]:
plt.figure(figsize=(13,5))
plt.bar(x_locs-bar_half, t1_revenue_total, width=bar_width, 
        label="Tydzień 1", tick_label=categories)
plt.bar(x_locs+bar_half, t2_revenue_total, width=bar_width, 
        label="Tydzień 2")
plt.legend()
plt.tight_layout()
plt.show()

Łączne operowanie na seriach

In [ ]:
plt.figure(figsize=(13,5))
bars_1 = plt.bar(x_locs-bar_half, t1_revenue_total, width=bar_width, label="Tydzień 1")
bars_2 = plt.bar(x_locs+bar_half, t2_revenue_total, width=bar_width, label="Tydzień 2")
plt.xticks(x_locs, categories)
pad_axis()
label_bars_pln(chain(bars_1, bars_2))
format_ytics_tys()

plt.legend()
plt.tight_layout()
plt.show()


Aktualizacja wyświetlania etykiet wartości

In [ ]:
def format_sep(x: float) -> str:
    return f"{int(round(x)):,}".replace(",", "\u202f")

In [ ]:
plt.figure(figsize=(13,5))
bars_1 = plt.bar(x_locs-bar_half, t1_revenue_total, width=bar_width, label="Tydzień 1", color="forestgreen")
bars_2 = plt.bar(x_locs+bar_half, t2_revenue_total, width=bar_width, label="Tydzień 2", color="steelblue")
plt.xticks(x_locs, categories)
pad_axis()
for b in bars_1:
    x, y = b.get_x(), b.get_height()
    plt.text(x, y, format_sep(y), ha="left", va="bottom")
for b in bars_2:
    x, y = b.get_x() + b.get_width(), b.get_height()
    plt.text(x, y, format_sep(y), ha="right", va="bottom")
format_ytics_tys()
plt.grid(axis="y", linestyle=":", alpha=0.6)
plt.title("LUMA - Porównanie przychodów per kategoria w tygodniu 1 i 2")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.legend()
plt.tight_layout()
plt.show()